In [2]:
import xml.dom.minidom as minidom
import xml.etree.ElementTree as ET
import klayout.db as db
import pandas as pd
import re

gds_path = r"C:\Users\malid\Downloads\malida_v4\malida_v4\LL_fun21_malida_combined_4.gds" # Path to your GDS file
csv_path = r"C:\Users\malid\Downloads\malida_v4\malida_v4\LL_fun21_malida_combined_4_results.csv"  # or qr5_LL_FUN21_results_2.csv / _3.csv

# 1. Detect actual top-cell name from your layout
layout = db.Layout()
layout.read(gds_path)

real_cell_name = layout.top_cell().name
print(f'Detected GDS top-cell name: {real_cell_name}')

# 2. Load CSV and keep only Class P physical errors
df = pd.read_csv(csv_path)
df_physical = df[~df['Check'].str.contains(r'\.DFM', regex=True)].copy()
df_physical['Category_ID'] = df_physical['Check'].str.replace(
    '.', '_', regex=False
)

# 3. Build XML Report Database
root = ET.Element('report-database')
ET.SubElement(root, 'description').text = 'MIT LL Class P Violations'
ET.SubElement(root, 'top-cell').text = real_cell_name

# Register cells block (MANDATORY for KLayout RDB)
cells_elem = ET.SubElement(root, 'cells')
cell_elem = ET.SubElement(cells_elem, 'cell')
ET.SubElement(cell_elem, 'name').text = real_cell_name

# Register categories (with dots converted to underscores)
categories_elem = ET.SubElement(root, 'categories')
for _, grp in df_physical.groupby(['Category_ID', 'Check']):
  cat_id = grp['Category_ID'].iloc[0]
  raw_rule = grp['Check'].iloc[0]
  cat = ET.SubElement(categories_elem, 'category')
  ET.SubElement(cat, 'name').text = str(cat_id)
  ET.SubElement(cat, 'description').text = str(raw_rule)

# Register violation items
items_elem = ET.SubElement(root, 'items')
for _, row in df_physical.iterrows():
  item = ET.SubElement(items_elem, 'item')
  ET.SubElement(item, 'cell').text = real_cell_name
  ET.SubElement(item, 'category').text = str(row['Category_ID'])
  ET.SubElement(item, 'visited').text = 'false'
  ET.SubElement(item, 'multiplicity').text = '1'

  pts = re.findall(r'\(([-0-9.]+)\s+([-0-9.]+)\)', str(row['Coordinates']))
  if pts:
    val = ET.SubElement(ET.SubElement(item, 'values'), 'value')
    val.text = f"polygon: ({';'.join(f'{x},{y}' for x, y in pts)})"

pretty_xml = minidom.parseString(ET.tostring(root, 'utf-8')).toprettyxml(
    indent='  '
)
with open(r'C:\Users\malid\Downloads\malida_v4\malida_v4\LL_fun21_malida_combined_4.lyrdb', 'w', encoding='utf-8') as f:
  f.write(pretty_xml)

print('Successfully exported USC_FUN21_combined_fix.lyrdb')




Detected GDS top-cell name: TOP
Successfully exported USC_FUN21_combined_fix.lyrdb


In [9]:
import os
import re
import xml.dom.minidom as minidom
import xml.etree.ElementTree as ET
import klayout.db as db

# ---------------------------------------------------------
# Configuration Paths
# ---------------------------------------------------------
gds_path = (
    r"C:\Users\malid\Downloads\USC_FUN21_combined_v2.gds"
)
txt_path = r"C:\Users\malid\Downloads\results.txt" # Path to your results3_1.txt error file
rdb_path = r"C:\Users\malid\Downloads\LL_fun21_usc_combined_v2_errors.lyrdb"

# ---------------------------------------------------------
# 1. Detect Real Top-Cell Name from GDS
# ---------------------------------------------------------
layout = db.Layout()
layout.read(gds_path)
real_cell_name = layout.top_cell().name
print(f"Detected GDS top-cell name: {real_cell_name}")

# ---------------------------------------------------------
# 2. Parse results3_1.txt for DRC Violations
# ---------------------------------------------------------
violations = []
current_rule = None
current_type = None
current_lines = []


def commit_block(rule, vtype, lines):
  if not rule or not lines or not vtype:
    return

  # Filter out .DFM checks if any are present
  if ".DFM" in rule:
    return

  if vtype == "polygon":
    pts = []
    for line in lines:
      m = re.search(r"\(\s*([-0-9.]+)\s*,\s*([-0-9.]+)\s*\)", line)
      if m:
        pts.append((m.group(1), m.group(2)))
    if pts:
      violations.append({"rule": rule, "type": "polygon", "points": pts})

  elif vtype == "edge":
    edges = []
    for line in lines:
      m = re.search(
          r"From\s*\(\s*([-0-9.]+)\s*,\s*([-0-9.]+)\s*\)\s*to\s*\(\s*([-0-9.]+)\s*,\s*([-0-9.]+)\s*\)",
          line,
      )
      if m:
        edges.append((m.group(1), m.group(2), m.group(3), m.group(4)))
    if edges:
      violations.append({"rule": rule, "type": "edge", "edges": edges})


with open(txt_path, "r", encoding="utf-8") as f:
  for raw_line in f:
    line = raw_line.strip()

    # Rule header
    rule_match = re.search(r"Rule:\s*(\S+)", line)
    if rule_match:
      commit_block(current_rule, current_type, current_lines)
      current_rule = rule_match.group(1)
      current_type = None
      current_lines = []
      continue

    # Violation header (Polygon or Edge)
    viol_match = re.search(r"(Polygon|Edge)\s+violation", line, re.IGNORECASE)
    if viol_match:
      commit_block(current_rule, current_type, current_lines)
      current_type = viol_match.group(1).lower()
      current_lines = []
      continue

    # Skip cell reference lines inside the block
    if line.startswith("GDS cell:"):
      continue

    if line:
      current_lines.append(line)

# Commit final block
commit_block(current_rule, current_type, current_lines)

# ---------------------------------------------------------
# 3. Build XML Report Database (.lyrdb)
# ---------------------------------------------------------
root = ET.Element("report-database")
ET.SubElement(root, "description").text = (
    "MIT LL SQP 3.0 DRC Violations (results3_1.txt)"
)
ET.SubElement(root, "top-cell").text = real_cell_name

# Register cells block (Mandatory for KLayout RDB)
cells_elem = ET.SubElement(root, "cells")
cell_elem = ET.SubElement(cells_elem, "cell")
ET.SubElement(cell_elem, "name").text = real_cell_name

# Register unique categories
unique_rules = sorted(list({v["rule"] for v in violations}))
categories_elem = ET.SubElement(root, "categories")
for rule in unique_rules:
  cat_id = rule.replace(".", "_")
  cat = ET.SubElement(categories_elem, "category")
  ET.SubElement(cat, "name").text = cat_id
  ET.SubElement(cat, "description").text = rule

# Register violation items
items_elem = ET.SubElement(root, "items")
for viol in violations:
  rule = viol["rule"]
  cat_id = rule.replace(".", "_")

  item = ET.SubElement(items_elem, "item")
  ET.SubElement(item, "cell").text = real_cell_name
  ET.SubElement(item, "category").text = cat_id
  ET.SubElement(item, "visited").text = "false"
  ET.SubElement(item, "multiplicity").text = "1"

  values_elem = ET.SubElement(item, "values")

  if viol["type"] == "polygon":
    poly_str = ";".join(f"{x},{y}" for x, y in viol["points"])
    val = ET.SubElement(values_elem, "value")
    val.text = f"polygon: ({poly_str})"

  elif viol["type"] == "edge":
    # If the check output pairs of interacting edges (e.g. edge-pair spacing check)
    if len(viol["edges"]) == 2:
      e1 = viol["edges"][0]
      e2 = viol["edges"][1]
      val = ET.SubElement(values_elem, "value")
      val.text = f"edge-pair: ({e1[0]},{e1[1]};{e1[2]},{e1[3]})/({e2[0]},{e2[1]};{e2[2]},{e2[3]})"
    else:
      for e in viol["edges"]:
        val = ET.SubElement(values_elem, "value")
        val.text = f"edge: ({e[0]},{e[1]};{e[2]},{e[3]})"

# Pretty-print XML and write to disk
pretty_xml = minidom.parseString(ET.tostring(root, "utf-8")).toprettyxml(
    indent="  "
)
with open(rdb_path, "w", encoding="utf-8") as f:
  f.write(pretty_xml)

print(
    f"Successfully parsed {len(violations)} violations and saved to: {rdb_path}"
)

Detected GDS top-cell name: TOP
Successfully parsed 82 violations and saved to: C:\Users\malid\Downloads\LL_fun21_usc_combined_v2_errors.lyrdb


In [3]:
def snap_rectangle_to_grid(
    coords, grid=0.005
) -> dict[str, tuple[float, float]]:
  """Shifts a rectangle to the nearest 5 nm grid while maintaining exact width and height.

  Parameters:
      coords: Either a list of 4 (x, y) vertices, OR a tuple/list of two points:
              [(x_ll, y_ll), (x_ur, y_ur)].
      grid (float): Grid size in micrometers (default 0.005 um = 5 nm).

  Returns:
      dict: Contains 'lower_left', 'upper_right', 'width', and 'height' in um.
  """
  pts = [tuple(map(float, pt)) for pt in coords]

  # Extract bounding box boundaries
  xs = [p[0] for p in pts]
  ys = [p[1] for p in pts]
  x_min, x_max = min(xs), max(xs)
  y_min, y_max = min(ys), max(ys)

  # Lock intentional dimensions
  width = round(x_max - x_min, 6)
  height = round(y_max - y_min, 6)
  w_locked = round(round(width / grid) * grid, 5)
  h_locked = round(round(height / grid) * grid, 5)

  # Shift lower-left corner to the nearest 5 nm grid point
  ll_x = round(round(x_min / grid) * grid, 5)
  ll_y = round(round(y_min / grid) * grid, 5)

  # Derive upper-right corner using the locked dimensions
  ur_x = round(ll_x + w_locked, 5)
  ur_y = round(ll_y + h_locked, 5)

  return {
      "lower_left": (ll_x, ll_y),
      "upper_right": (ur_x, ur_y),
      "width": w_locked,
      "height": h_locked,
  }




In [31]:
x1=1881.97500
y1 =-3190.50500
x2 =1883.47500
y2 =-3190.25500
corners_box2 = [
    (x1,y1),  # Lower-left (3.5 nm off-grid)
    (x2,y2),  # Upper-right
]

res2 = snap_rectangle_to_grid(corners_box2)
# print("--- Result for Box 2 (image_8b4d1d.png) ---")
print(f"Lower Left : x = {res2['lower_left'][0]:.5f}, y = {res2['lower_left'][1]:.5f}")
print(f"Upper Right: x = {res2['upper_right'][0]:.5f}, y = {res2['upper_right'][1]:.5f}")
print(f"Dimensions : W = {res2['width']:.5f} um, H = {res2['height']:.5f} um")

Lower Left : x = 1881.97500, y = -3190.50500
Upper Right: x = 1883.47500, y = -3190.25500
Dimensions : W = 1.50000 um, H = 0.25000 um
